In [19]:

from langgraph.graph import StateGraph, START, END
from typing import TypedDict, Annotated
from langchain_core.messages import BaseMessage, HumanMessage
from langchain_openai import ChatOpenAI
from langgraph.checkpoint.memory import MemorySaver
from langgraph.graph.message import add_messages


In [20]:
def model_name():
    return ChatOpenAI(model="gpt-4o-mini")

model = model_name()

In [21]:
class ChatState(TypedDict):
    messages: Annotated[list[BaseMessage], add_messages]

In [22]:
def chat_node(state: ChatState) -> dict:
    messages = state["messages"]
    response = model.invoke(messages)
    return {"messages": [response]}

In [29]:
builder = StateGraph(ChatState)

builder.add_node("chat_node", chat_node)

builder.add_edge(START, "chat_node")
builder.add_edge("chat_node", END)

checkpointer = MemorySaver()

chatbot = builder.compile(checkpointer=checkpointer)

In [30]:
if __name__ == "__main__":
    config = {"configurable": {"thread_id": "chat-1"}}

    while True:
        user_input = input("You: ")
        print(f"You: {user_input}")
        if user_input.lower() in ["exit", "quit"]:
            break

        result = chatbot.invoke(
            {"messages": [HumanMessage(content=user_input)]},
            config=config
        )

        print("AI:", result["messages"][-1].content)

You: Hi
AI: Hello! How can I assist you today?
You: What you can do for me
AI: I can help with a variety of things, including:

1. **Answering Questions**: I can provide information on a wide range of topics, from science and history to technology and pop culture.
2. **Writing Assistance**: Whether you need help drafting emails, essays, or creative writing, I can offer guidance and suggestions.
3. **Learning and Education**: I can help explain concepts, solve problems, and offer study tips.
4. **Recommendations**: I can suggest books, movies, music, or products based on your interests.
5. **General Advice**: I can provide tips on various topics, including health, productivity, and personal development.

Let me know what you need help with!
You: my name is sohail
AI: Nice to meet you, Sohail! How can I assist you today?
You: what is my name
AI: Your name is Sohail. How can I help you today, Sohail?
You: exit


In [32]:
from pprint import pprint

In [33]:
state = chatbot.get_state(config)
pprint(state.values["messages"])

[HumanMessage(content='Hi', additional_kwargs={}, response_metadata={}, id='e03d1944-9683-4cce-9fe6-8e85117d8db6'),
 AIMessage(content='Hello! How can I assist you today?', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 9, 'prompt_tokens': 8, 'total_tokens': 17, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0, 'text_tokens': None}, 'prompt_tokens_details': {'audio_tokens': 0, 'cache_write_tokens': None, 'cached_tokens': 0, 'image_tokens': None, 'text_tokens': None}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_4adbad45a5', 'id': 'chatcmpl-ENyX51U81plc1qCtOyseoGXfrYzvp', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--01a09f88-ccf3-7d52-9e26-8f4a0744e604-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 8, 'output_tokens': 9, 'total_tokens': 17, '